In [1]:
import numpy as np
import sys
from pathlib import Path
PROJECT_ROOT = Path("../").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.features.sbert_features import SBERTFeatureExtractor
from src.utils.utils import load_csv
from src.config import (
    CLEANED_TRAIN_PATH,
    CLEANED_TEST_PATH,
    TRAIN_LABEL_PATH,
    FINAL_TRAIN_SBERT_PURE_PATH,
    FINAL_TEST_SBERT_PURE_PATH
)


# Load data
train_df = load_csv(CLEANED_TRAIN_PATH)
test_df = load_csv(CLEANED_TEST_PATH)
y_train = load_csv(TRAIN_LABEL_PATH)

# Convert label (nếu cần)
y_train_pure = (y_train - 1).values.ravel()

# ===== SBERT ONLY =====
sbert = SBERTFeatureExtractor()

print("🚀 Encoding TRAIN with SBERT only...")
X_train_pure = sbert.transform(train_df)

print("🚀 Encoding TEST with SBERT only...")
X_test_pure = sbert.transform(test_df)

# Save dataset
np.save(FINAL_TRAIN_SBERT_PURE_PATH, X_train_pure)
np.save(FINAL_TEST_SBERT_PURE_PATH, X_test_pure)

print("✔ Saved SBERT-only dataset!")
print("Train shape:", X_train_pure.shape)
print("Test shape:", X_test_pure.shape)

/Users/nhatnam/Documents/DM_252/Assignment/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded SPECTER2 model: allenai/specter2_base on cpu
🚀 Encoding TRAIN with SBERT only...
🚀 Encoding TEST with SBERT only...
✔ Saved SBERT-only dataset!
Train shape: (510, 768)
Test shape: (86, 768)


In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from src.config import SUBMISSION_PURE_SBERT_PATH


# 3. Logistic Regression cơ bản nhất
baseline_model = LogisticRegression(class_weight='balanced', max_iter=2000)

# 4. Kiểm tra CV
cv_scores = cross_val_score(baseline_model, X_train_pure, y_train_pure, cv=5, scoring='f1_macro')
print(f"✔ PURE SBERT CV Macro F1-Score: {cv_scores.mean():.4f}")

# 5. Predict & Submit
baseline_model.fit(X_train_pure, y_train_pure)
y_pred_baseline = baseline_model.predict(X_test_pure) + 1

submission_base = pd.DataFrame({
    "id": test_df["id"],
    "Label": y_pred_baseline
})
submission_base.to_csv(SUBMISSION_PURE_SBERT_PATH, index=False)
print("✔ Saved PURE baseline submission")

✔ PURE SBERT CV Macro F1-Score: 0.3001
✔ Saved PURE baseline submission


In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

print(f"\n====================")
print(f"MODEL: SBERT + OPTIMIZED PCA + SVM")
print(f"====================")

# 1. Pipeline hoàn chỉnh: Chuẩn hóa -> Giảm chiều -> Phân loại
pipe = Pipeline([
    ('scaler', StandardScaler()), # Chuẩn hóa giúp PCA tìm thành phần chính chuẩn xác hơn
    ('pca', PCA(random_state=42)),
    ('clf', SVC(class_weight='balanced', random_state=42))
])

# 2. Định nghĩa Lưới tham số (Grid) để máy tính tự dò tìm
param_grid = {
    'pca__n_components': [30, 50, 80, 120, 200], # Quét mọi mức cắt giảm chiều
    'clf__kernel': ['linear', 'rbf'],            # Thử cả ranh giới tuyến tính và phi tuyến
    'clf__C': [0.5, 1.0, 5.0, 10.0]              # Mức độ phạt khi mô hình phân loại sai
}

# 3. Khởi tạo và chạy Grid Search (Quét 5 x 2 x 4 = 40 kịch bản)
print("✔ Đang tự động dò tìm cấu trúc PCA tốt nhất (vui lòng đợi vài giây)...")
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1)
grid.fit(X_train_pure, y_train_pure)

print(f"\n✔ Best CV Macro F1-Score: {grid.best_score_:.4f}")
print(f"✔ Best Params Found: {grid.best_params_}")

# 4. Lưu lại submission từ mô hình tốt nhất vừa tìm được
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test_pure) + 1 # Nhớ cộng 1

submission_best_pca = pd.DataFrame({
    "id": test_df["id"],
    "Label": y_pred_best
})
submission_best_pca.to_csv("submission_sbert_best_pca.csv", index=False)
print("✔ Saved Optimized SBERT + PCA submission")


MODEL: SBERT + OPTIMIZED PCA + SVM
✔ Đang tự động dò tìm cấu trúc PCA tốt nhất (vui lòng đợi vài giây)...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

✔ Best CV Macro F1-Score: 0.3221
✔ Best Params Found: {'clf__C': 1.0, 'clf__kernel': 'linear', 'pca__n_components': 30}
✔ Saved Optimized SBERT + PCA submission


In [3]:
# ============================================
# Evaluate LLM ASP-Relevance Scoring using Ollama
# ============================================

import pandas as pd
import ollama
import json

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

from tqdm import tqdm

import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.config import CLEANED_TRAIN_PATH

# ======================
# Load train data
# ======================

train_df = pd.read_csv(CLEANED_TRAIN_PATH)

print(train_df.columns)

# ======================
# Better SYSTEM PROMPT
# ======================

SYSTEM_PROMPT = """
You are an expert in Answer Set Programming (ASP),
logic programming,
knowledge representation,
nonmonotonic reasoning,
SAT solving,
and constraint solving.

Your task is to rate how relevant a research paper is
to ASP researchers.

Scoring rubric:

1 = Completely unrelated to ASP or logic programming
Examples:
- Computer vision
- CNN image classification
- Generic NLP
- Networking
- Databases
- Bioinformatics

2 = Slightly related to AI/reasoning but not ASP-focused
Examples:
- General machine learning
- Data mining
- Generic optimization
- Reinforcement learning

3 = Relevant to ASP researchers but not central ASP
Examples:
- SAT solving
- SMT solving
- Constraint programming
- Planning
- Automated reasoning

4 = Strongly relevant to ASP
Examples:
- Logic programming
- Stable model semantics
- ASP grounding
- Nonmonotonic reasoning
- ASP systems

5 = Core ASP contribution
Examples:
- New ASP semantics
- ASP solver advances
- ASP optimization techniques
- Foundational ASP theory
- ASP competitions
- ASP benchmarks

Example 1:
Title: Deep Residual Networks for Image Recognition
Score: 1

Example 2:
Title: Constraint Satisfaction Techniques in AI Planning
Score: 3

Example 3:
Title: Advances in Stable Model Semantics for ASP
Score: 5

Important:
- SAT/SMT/CSP papers are usually 3 unless tightly connected to ASP.
- Logic programming papers are usually 4-5.
- Machine learning papers are usually 1-2.

Respond ONLY with valid JSON.

Format:
{"score": <1-5>, "reason": "<short reason>"}
"""

# ======================
# Query Ollama
# ======================

def score_paper(title, abstract):

    prompt = f"""
Title: {title}

Abstract: {abstract}
"""

    try:

        response = ollama.chat(
            model="qwen2.5:7b",
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            options={
                "temperature": 0
            }
        )

        text = response["message"]["content"].strip()

        # Debug raw output
        print("\nRAW RESPONSE:")
        print(text)

        # Remove markdown fences if model adds them
        text = text.replace("```json", "")
        text = text.replace("```", "")
        text = text.strip()

        result = json.loads(text)

        score = int(result["score"])

        # Clamp score to 1-5
        score = max(1, min(5, score))

        return {
            "score": score,
            "reason": result["reason"]
        }

    except Exception as e:

        print("ERROR:", e)

        return {
            "score": None,
            "reason": str(e)
        }

# ======================
# Run on train set
# ======================

N_SAMPLES = 50

subset_df = train_df.head(N_SAMPLES).copy()

llm_scores = []
llm_reasons = []

for _, row in tqdm(subset_df.iterrows(), total=len(subset_df)):

    title = str(row["title"])
    abstract = str(row["abstract"])

    result = score_paper(title, abstract)

    llm_scores.append(result["score"])
    llm_reasons.append(result["reason"])

# ======================
# Save predictions
# ======================

subset_df["llm_score"] = llm_scores
subset_df["llm_reason"] = llm_reasons

# ======================
# Evaluate MULTICLASS
# ======================

valid_df = subset_df[
    subset_df["llm_score"].notna()
]

y_true = valid_df["Label"].astype(int)
y_pred = valid_df["llm_score"].astype(int)

print("\n====================")
print("LLM Evaluation")
print("====================")

print("Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        digits=3
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# ======================
# Wrong predictions
# ======================

wrong_df = valid_df[
    valid_df["Label"] != valid_df["llm_score"]
]

print("\n====================")
print("Wrong Predictions")
print("====================")

display(
    wrong_df[
        [
            "title",
            "Label",
            "llm_score",
            "llm_reason"
        ]
    ].head(20)
)

# ======================
# Save
# ======================

subset_df.to_csv(
    "train_llm_scored.csv",
    index=False
)

print("\nSaved to train_llm_scored.csv")

Index(['title', 'venue', 'year', 'authors', 'doi', 'Label', 'id', 'abstract'], dtype='object')


  2%|▏         | 1/50 [00:03<02:35,  3.18s/it]


RAW RESPONSE:
{"score": 3, "reason": "Relevant to ASP researchers as it deals with rule languages and reasoning over streams, but not central to ASP."}


  4%|▍         | 2/50 [00:05<02:01,  2.52s/it]


RAW RESPONSE:
{"score": 4, "reason": "Relevant to ASP researchers as it uses Answer Set Programming and focuses on legal reasoning, but not a core ASP contribution."}


...


RAW RESPONSE:
{"score": 1, "reason": "Completely unrelated to ASP or logic programming"}


100%|██████████| 50/50 [01:29<00:00,  1.80s/it]


RAW RESPONSE:
{"score": 3, "reason": "Relevant to ASP researchers as it involves planning and timed automata, but not specifically focused on ASP."}

LLM Evaluation
Accuracy: 0.2

Classification Report:
              precision    recall  f1-score   support

           1      0.333     0.143     0.200        14
           2      0.500     0.167     0.250         6
           3      0.174     0.500     0.258         8
           4      0.167     0.273     0.207        11
           5      0.000     0.000     0.000        11

    accuracy                          0.200        50
   macro avg      0.235     0.216     0.183        50
weighted avg      0.218     0.200     0.173        50


Confusion Matrix:
[[2 1 8 2 1]
 [2 1 1 2 0]
 [0 0 4 4 0]
 [1 0 7 3 0]
 [1 0 3 7 0]]

Wrong Predictions


,title,Label,llm_score,llm_reason
0,tabled clp for reasoning over stream data,1,3,Relevant to ASP researchers as it deals with r...
1,xai-law: a logic programming tool for modeling...,5,4,Relevant to ASP researchers as it uses Answer ...
2,workshop proceedings of the 40th international...,1,3,Focuses on argumentation theory and frameworks...
3,probabilistic active goal recognition,5,3,Relevant to AI/active reasoning but not centra...
4,heuristic strategies for accelerating multi-ag...,1,3,Relevant to planning but not central to ASP
6,action languages and covid-19: lessons learned,2,1,Not related to ASP or logic programming.
7,flexible lifelong explainable and robust solut...,4,3,Relevant to AI planning but not central to ASP
8,planning domain model acquisition from state t...,1,3,Relevant to ASP researchers as it deals with p...
9,chrvis: syntax and semantics,1,4,"Introduces a new syntax and semantics for ASP,..."
10,exploring neurosymbolic systems in answer set ...,5,4,Relevant to ASP researchers as it applies ASP ...



Saved to train_llm_scored.csv
